# Decode the Distance: Logical-Memory Decoding with Stim and PyMatching

**Contributor:** Shalini Devendrababu

This challenge is about decoding a small logical-memory experiment. You will generate surface-code memory circuits with Stim, build a minimum-weight matching decoder with PyMatching, estimate logical error rates, and test what happens when the decoder uses the wrong noise model.

The central question is concrete:

> For a fixed circuit-level noise model and shot budget, when does increasing the code distance reduce the logical error rate?


## Setup

Install the local requirements from this folder:

```bash
!python -m pip install -r requirements.txt
```

No GPU, cloud account, API token, or quantum hardware is needed.


In [ ]:
from __future__ import annotations

from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pymatching
import stim

import grader

SEED = 2026


## Background

A logical memory repeatedly measures stabilizers while trying to preserve one logical qubit. Stim can compile a circuit-level noise model into a **detector error model**. A detector event marks a parity-check change that should not happen without faults. PyMatching converts the detector error model into a graph and decodes each sampled syndrome.

For each shot, Stim returns detector events and logical observable flips. The decoder succeeds when its predicted logical flip matches the actual observable flip.


## Task 1 — Warm-up: decode a repetition-code syndrome

For a bit-flip repetition code with data-error vector `e`, the adjacent-check syndrome is

```text
s_i = e_i xor e_{i+1}.
```

Implement two small functions before using Stim.

**Input/output contracts**

```text
repetition_syndrome(error) -> np.ndarray of shape (len(error)-1,)
decode_repetition_syndrome(syndrome) -> binary correction of shape (len(syndrome)+1,)
```

The correction should have the requested syndrome and minimum Hamming weight among the two possible chains.


In [ ]:
def repetition_syndrome(error: np.ndarray) -> np.ndarray:
    # TODO: return adjacent xor checks as uint8 values.
    raise NotImplementedError("Task 1: implement repetition_syndrome")


def decode_repetition_syndrome(syndrome: np.ndarray) -> np.ndarray:
    # TODO: construct the two candidate chains and return a minimum-weight one.
    raise NotImplementedError("Task 1: implement decode_repetition_syndrome")


In [ ]:
try:
    print(grader.check_repetition_syndrome(repetition_syndrome))
    print(grader.check_repetition_decoder(decode_repetition_syndrome))
except NotImplementedError as exc:
    print("TODO expected:", exc)


## Task 2 — Generate a rotated surface-code memory circuit

Implement `make_memory_circuit`.

**Input contract**

```text
make_memory_circuit(distance, rounds, p, basis="z") -> stim.Circuit
```

- `distance`: odd integer at least 3;
- `rounds`: positive integer;
- `p`: uniform circuit-level noise rate in `[0, 0.5)`;
- `basis`: either `"x"` or `"z"`.

Use Stim's generated circuits: `surface_code:rotated_memory_z` or `surface_code:rotated_memory_x`. Set these four noise parameters to `p`:

```text
before_round_data_depolarization
after_clifford_depolarization
after_reset_flip_probability
before_measure_flip_probability
```

**Public check:** for distance 3, rounds 3, z-basis memory, the circuit has 24 detectors and one logical observable.


In [ ]:
def make_memory_circuit(distance: int, rounds: int, p: float, basis: str = "z") -> stim.Circuit:
    # TODO: validate inputs and call stim.Circuit.generated.
    raise NotImplementedError("Task 2: implement make_memory_circuit")


In [ ]:
try:
    print(grader.check_make_memory_circuit(make_memory_circuit))
except NotImplementedError as exc:
    print("TODO expected:", exc)


## Task 3 — Build a matching decoder and decode samples

Implement the core simulation and decoding functions.

**Input/output contracts**

```text
build_matching(circuit, weighted=True, uniform_probability=0.01) -> pymatching.Matching
sample_and_decode(circuit, matching, shots, seed) -> (detector_events, observable_flips, predictions)
logical_failures(predictions, observable_flips) -> Boolean vector of length shots
```

Hints:

- Use `circuit.detector_error_model(decompose_errors=True)`.
- Use `pymatching.Matching.from_detector_error_model(...)`.
- To make an intentionally naive unweighted decoder, replace every `error(p)` in the detector error model by `error(uniform_probability)` before building the matching graph. The helper `grader.replace_detector_error_probabilities` does this.
- Use `circuit.compile_detector_sampler(seed=seed).sample(..., separate_observables=True)`.


In [ ]:
def build_matching(
    circuit: stim.Circuit,
    *,
    weighted: bool = True,
    uniform_probability: float = 0.01,
) -> pymatching.Matching:
    # TODO: build a detector error model and return a pymatching.Matching.
    raise NotImplementedError("Task 3: implement build_matching")


def sample_and_decode(
    circuit: stim.Circuit,
    matching: pymatching.Matching,
    shots: int,
    seed: int,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    # TODO: sample detector events and observable flips, then decode in a batch.
    raise NotImplementedError("Task 3: implement sample_and_decode")


def logical_failures(predictions: np.ndarray, observable_flips: np.ndarray) -> np.ndarray:
    # TODO: return a Boolean vector whose entries are True on failed shots.
    raise NotImplementedError("Task 3: implement logical_failures")


In [ ]:
try:
    print(grader.check_build_matching(build_matching))
    print(grader.check_sample_and_decode(sample_and_decode))
    print(grader.check_logical_failures(logical_failures))
except NotImplementedError as exc:
    print("TODO expected:", exc)


## Task 4 — Estimate a logical error rate with uncertainty

A logical error rate is a binomial proportion. Report uncertainty, not only a point estimate.

**Input/output contracts**

```text
wilson_interval(failures, shots, z=1.959963984540054) -> (rate, lower, upper)
estimate_logical_error_rate(circuit, shots, seed, decoder_circuit=None, weighted=True) -> dict
```

If `decoder_circuit` is `None`, sample from and decode with the same circuit model. If it is supplied, sample from `circuit` but build the decoder from `decoder_circuit`. This supports the mismatch test in Task 5.

Return a dictionary with at least these keys:

```text
shots, failures, rate, ci_low, ci_high, num_detectors, num_observables, num_qubits
```


In [ ]:
def wilson_interval(
    failures: int,
    shots: int,
    z: float = 1.959963984540054,
) -> tuple[float, float, float]:
    # TODO: return (rate, lower, upper) for a Wilson score interval.
    raise NotImplementedError("Task 4: implement wilson_interval")


def estimate_logical_error_rate(
    circuit: stim.Circuit,
    shots: int,
    seed: int,
    *,
    decoder_circuit: stim.Circuit | None = None,
    weighted: bool = True,
) -> dict[str, Any]:
    # TODO: build a decoder, sample, decode, count failures, and add interval fields.
    raise NotImplementedError("Task 4: implement estimate_logical_error_rate")


In [ ]:
try:
    print(grader.check_wilson_interval(wilson_interval))
    print(grader.check_zero_noise_pipeline(make_memory_circuit, estimate_logical_error_rate))
except NotImplementedError as exc:
    print("TODO expected:", exc)


## Task 5 — Distance scaling and decoder mismatch

Run two experiments.

**Experiment A: distance scaling**

Recommended defaults:

```text
distances = [3, 5, 7]
p_values = [0.001, 0.003, 0.006, 0.01, 0.02]
shots = 2000 or 3000
rounds = distance
```

Plot logical error rate versus physical error rate with 95% Wilson intervals. A good answer explains where higher distance helps and where it does not.

**Experiment B: mismatch stress test**

Sample data from a circuit with asymmetric noise, but decode it three ways:

1. decoder built from the true asymmetric model;
2. decoder built from a uniform noise model;
3. unweighted decoder that keeps the graph but ignores probability weights.

Report logical error rates and intervals for all three.


In [ ]:
# TODO: run the distance-scaling benchmark.
# TODO: plot logical error rate with interval bars.
# TODO: run the mismatch stress test and explain the result.


## Bonus tasks

- Compare rotated and unrotated memory circuits at similar qubit budgets.
- Add a biased-noise model and check when z-basis and x-basis memories behave differently.
- Fit a small threshold-style crossing plot, with a warning about finite-size effects.
- Try a non-matching decoder or a small learned post-processor, and compare accuracy and runtime.
- Use importance sampling or repeated runs to estimate rare logical failures more efficiently.

## Submission guidance

A strong submission should include working code, plots, a short interpretation of distance scaling, a mismatch experiment, and a few sentences about failure modes.


## References

- Stim documentation and getting-started notebook.
- PyMatching documentation.
- Higgott and Gidney, *Sparse Blossom: correcting a million errors per core second with minimum-weight matching*, Quantum 9, 1600 (2025).
- Terhal, *Quantum error correction for quantum memories*, Rev. Mod. Phys. 87, 307 (2015).
